In [1]:
# Install the Hugging Face datasets library
!pip install datasets pandas

In [1]:
import pandas as pd
from datasets import load_dataset

# 1. Download
print("Downloading dataset...")
raw_dataset = load_dataset("Hello-SimpleAI/hc3", "default")
df = pd.DataFrame(raw_dataset['train'])

# 2. Flatten (Using your logic, but slightly faster with 'explode')
# This takes the lists in human_answers and chatgpt_answers and makes them individual rows
human_df = df[['human_answers']].explode('human_answers').rename(columns={'human_answers': 'text'})
human_df['label'] = 0

ai_df = df[['chatgpt_answers']].explode('chatgpt_answers').rename(columns={'chatgpt_answers': 'text'})
ai_df['label'] = 1

# 3. Combine and Shuffle
final_df = pd.concat([human_df, ai_df], ignore_index=True)
final_df = final_df.dropna() # Remove any empty rows
final_df = final_df.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Final Dataset Shape: {final_df.shape}")
final_df.head()

README.md: 0.00B [00:00, ?B/s]

all.jsonl:   0%|          | 0.00/73.7M [00:00<?, ?B/s]

finance.jsonl: 0.00B [00:00, ?B/s]

medicine.jsonl: 0.00B [00:00, ?B/s]

open_qa.jsonl: 0.00B [00:00, ?B/s]

reddit_eli5.jsonl:   0%|          | 0.00/55.4M [00:00<?, ?B/s]

wiki_csai.jsonl: 0.00B [00:00, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Final Dataset Shape: (170898, 2)


,text,label
0,"Simplest way to explain it . Yes , it is bad c...",0
1,"they not only file personal taxes , they help ...",0
2,"Cashmere wool, usually simply known as cashmer...",0
3,It's hard to say exactly where the idea to smo...,1
4,Depends on how you measure liquidity. There's...,0


In [2]:
import re

def clean_text(text):
    # Remove common AI prefixes
    prefixes = ["As an AI language model", "I am an AI", "Certainly!", "Sure, I can help"]
    for p in prefixes:
        text = text.replace(p, "")

    # Remove extra whitespaces and newlines
    text = re.sub(r'\s+', ' ', text).strip()
    return text

final_df['text'] = final_df['text'].apply(clean_text)
# Remove very short answers (less than 10 words) as they are hard to classify
final_df = final_df[final_df['text'].str.split().str.len() > 10]

Feature Extraction

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

# Split data: 80% Train, 20% Test
X_train, X_test, y_train, y_test = train_test_split(
    final_df['text'], final_df['label'], test_size=0.2, random_state=42
)

# Initialize TF-IDF (using n-grams as per your objectives)
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print("TF-IDF Matrix created.")

TF-IDF Matrix created.


Baseline model - Logistic Regression

In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

# Train model
model = LogisticRegression()
model.fit(X_train_tfidf, y_train)

# Predict
y_pred = model.predict(X_test_tfidf)

# Evaluate
print(f"Accuracy: {accuracy_score(y_test, y_pred)}")
print(classification_report(y_test, y_pred))

Accuracy: 0.9771912675138481
              precision    recall  f1-score   support

           0       0.98      0.99      0.98     23062
           1       0.97      0.96      0.96     10697

    accuracy                           0.98     33759
   macro avg       0.98      0.97      0.97     33759
weighted avg       0.98      0.98      0.98     33759



Linguistic Pattern Analysis

In [5]:
import numpy as np

# Get feature names from TF-IDF
feature_names = np.array(tfidf.get_feature_names_out())

# Get the coefficients from the Logistic Regression model
# (Positive coefficients = AI-like, Negative = Human-like)
coefficients = model.coef_[0]

# Get indices of top 20 and bottom 20
top_20_ai_indices = np.argsort(coefficients)[-20:]
top_20_human_indices = np.argsort(coefficients)[:20]

print("Top 20 AI-indicator words:", feature_names[top_20_ai_indices])
print("Top 20 Human-indicator words:", feature_names[top_20_human_indices])

Top 20 AI-indicator words: ['when you' 'helps' 'might' 'sure' 'doesn' 'overall' 'have any' 'nthe'
 'because they' 'does that' 'don' 'example if' 'such as' 'it important'
 'means that' 'help' 'and' 'important' 'including' 'important to']
Top 20 Human-indicator words: ['url_0' 'etc' 'basically' 'ca' 'only' 'my' 'do' 'can not' 'but' 'most'
 'thus' 'probably' 'pretty' 'those' 'now' 'all' 'either' 'edit' 'an'
 'what']


SVM and Random Forest

In [6]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC

# Try SVM
svm_model = LinearSVC()
svm_model.fit(X_train_tfidf, y_train)
print(f"SVM Accuracy: {svm_model.score(X_test_tfidf, y_test)}")

# Try Random Forest (might take longer)
rf_model = RandomForestClassifier(n_estimators=100)
rf_model.fit(X_train_tfidf, y_train)
print(f"Random Forest Accuracy: {rf_model.score(X_test_tfidf, y_test)}")

SVM Accuracy: 0.986018543203294
Random Forest Accuracy: 0.992535323913623


In [7]:
!pip install transformers[torch] accelerate -U

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 61.9 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [8]:
from transformers import AutoTokenizer, DataCollatorWithPadding
import torch
from torch.utils.data import Dataset

# Use DistilRoBERTa - a faster, lighter version of RoBERTa
model_name = "distilroberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.encodings = tokenizer(list(texts), truncation=True, padding=True, max_length=max_len)
        self.labels = list(labels)

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

# Take a smaller subset (e.g., 5000 rows) to train quickly in Colab
subset_df = final_df.sample(5000, random_state=42)
train_texts, val_texts, train_labels, val_labels = train_test_split(subset_df['text'], subset_df['label'], test_size=0.2)

train_dataset = TextDataset(train_texts, train_labels, tokenizer)
val_dataset = TextDataset(val_texts, val_labels, tokenizer)

config.json:   0%|          | 0.00/480 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [12]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    eval_strategy="epoch",
    logging_steps=10,
    learning_rate=2e-5,
    weight_decay=0.01,
    save_strategy="epoch",
    load_best_model_at_end=True
)

In [13]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

trainer.train()

Epoch,Training Loss,Validation Loss
1,0.044847,0.093943
2,0.000213,0.091815
3,0.000116,0.065152


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=750, training_loss=0.04781296232994646, metrics={'train_runtime': 190.1812, 'train_samples_per_second': 63.098, 'train_steps_per_second': 3.944, 'total_flos': 397402195968000.0, 'train_loss': 0.04781296232994646, 'epoch': 3.0})

In [16]:
# Run this in a new cell
eval_results = trainer.evaluate()
print(f"Final Validation Loss: {eval_results['eval_loss']:.4f}")

Training Loss,Validation Loss,Epoch
0.000116,0.065152,3


Final Validation Loss: 0.0652


In [22]:
def predict_text(text):
    # Prepare the text
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=128).to("cuda")

    # Get prediction
    model.eval()
    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.nn.functional.softmax(outputs.logits, dim=-1)
        prediction = torch.argmax(probs, dim=-1).item()
        confidence = probs[0][prediction].item()

    label = "AI-Generated" if prediction == 1 else "Human-Authored"
    return f"Result: {label} (Confidence: {confidence:.2%})"

# --- TEST IT ---
my_text = "The City of Light has been the heart of Western culture for centuries. A few things that make it endlessly fascinating:"
print(predict_text(my_text))

Result: AI-Generated (Confidence: 99.99%)


In [23]:
test_human = "Honestly, I think Paris is okay but it's super expensive and way too many tourists everywhere. The bread was cool though."
print(f"Human Test -> {predict_text(test_human)}")

Human Test -> Result: Human-Authored (Confidence: 99.98%)


In [24]:
test_adversarial = "Paris is like the capital of France and totally one of the most iconic cities ever. It blends history and art n culture."
print(f"Adversarial Test -> {predict_text(test_adversarial)}")

Adversarial Test -> Result: Human-Authored (Confidence: 99.98%)


In [25]:
import numpy as np
from sklearn.metrics import accuracy_score, classification_report

# Get predictions for the whole validation set
print("Generating predictions for validation set... (this takes a minute)")
raw_predictions = trainer.predict(val_dataset)
y_preds = np.argmax(raw_predictions.predictions, axis=-1)
y_true = raw_predictions.label_ids

# Final results
acc = accuracy_score(y_true, y_preds)
print(f"\nDistilRoBERTa Accuracy: {acc:.4f}")
print("\nDetailed Comparison Report:")
print(classification_report(y_true, y_preds, target_names=["Human", "AI"]))

Generating predictions for validation set... (this takes a minute)



DistilRoBERTa Accuracy: 0.9860

Detailed Comparison Report:
              precision    recall  f1-score   support

       Human       1.00      0.98      0.99       666
          AI       0.96      1.00      0.98       334

    accuracy                           0.99      1000
   macro avg       0.98      0.99      0.98      1000
weighted avg       0.99      0.99      0.99      1000

